
Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.
Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [6]:
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:qwen/qwen3-32b")

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")


In [8]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7f5871c3e490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f5871c3ed50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 

In [9]:


model.invoke("Provide details about the moview Inception")



AIMessage(content='<think>\nOkay, I need to provide details about the movie "Inception." Let me start by recalling what I know. It\'s a 2010 film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, right? He plays Dom Cobb, a thief who steals information by infiltrating the subconscious of his targets. The movie is known for its complex plot involving dreams within dreams and the concept of planting an idea, called "inception." \n\nI should mention the cast. There\'s Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. Each has significant roles. The plot revolves around a team entering different layers of dreams to plant an idea. There\'s also a lot of action, like the hotel fight scene or the rotating hallway fight. The music is by Hans Zimmer, which is iconic, with the use of a slowed-down version of Édith Piaf\'s "Non, Je Ne Regrette Rien." \n\nThe film\'s narrative structure is non-linear, with multiple timelines and layers. The ending is ambiguous, with the sp

In [10]:


response=model_with_structure.invoke("Provide details about the moview Inception")
response



Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

Message output alongside parsed structure

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to fill those parameters. I remember Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the year and director. Yep, 2010 and Christopher Nolan. The rating is 8.8. So I'll structure the tool call with those details. Make sure all required fields are included. No need for extra info, just the parameters from the function. Alright, that should cover it.\n", 'tool_calls': [{'id': 'sj60pyd0k', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 190, 'prompt_tokens': 231, 'total_tokens': 421, 'completion_time': 0.379026394, 'c

Nested Structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Cyril')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

In [13]:
model.invoke("what is 2+2")

AIMessage(content='<think>\nOkay, let me try to figure out what 2+2 is. Hmm, I remember from school that adding numbers together means combining their quantities. So 2 plus 2... if I have two apples and add another two apples, how many do I have in total? Let me count: 1, 2, then 3, 4. So that\'s four apples. Wait, is that right? Maybe I should visualize it. Two blocks plus two blocks would make four blocks. Yeah, that seems to make sense.\n\nBut wait, could there be a trick here? Sometimes questions are meant to be simple but have a hidden complexity. Like, maybe in different number bases? For example, in base 3, 2+2 would be 11 because 2+2 is 4 in decimal, and 4 divided by 3 is 1 with a remainder of 1. So in base 3, it would be written as 11. But the question just says "2+2" without specifying the base, so I think the default is base 10. In base 10, 2+2 is definitely 4. \n\nAnother angle: maybe in some contexts, like in a joke or a riddle, the answer isn\'t 4. There\'s a famous joke 

TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [15]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the bhulbhulayiya3")
response

{'director': 'Anees Bazmee',
 'rating': 6.5,
 'title': 'Bhool Bhulaiyaa 3',
 'year': 2022}

In [19]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie three ediots")
response

{'budget': 10000000,
 'cast': [{'name': 'Aamir Khan', 'role': 'Raju'},
  {'name': 'R. Madhavan', 'role': 'Krishna'},
  {'name': 'Sharman Joshi', 'role': 'Vardhan'}],
 'genres': ['Comedy', 'Drama'],
 'title': 'Three Idiots',
 'year': 2009}

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie bahubali")# if the llm get confused and asks for clarification this block of code will generate error
response                                                                         #so the prompt must be clear as the structure otuput will not match

{'budget': 100000000,
 'cast': [{'name': 'Prabhas', 'role': 'Bahubali'},
  {'name': 'Sridevi', 'role': 'Queen Sivagami'},
  {'name': 'Tamannaah', 'role': 'Devasena'}],
 'genres': ['Action', 'Adventure', 'Fantasy'],
 'title': 'Bahubali: The Beginning',
 'year': 2015}

In [21]:
model_with_structure.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [22]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

In [ ]:
result["structured_response"]

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
